In [ ]:

import numpy as np

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE, RFECV
from sklearn.model_selection import cross_val_score


# ==========================================
# STEP 1: GENERATE DATA
# ==========================================

def make_data(seed=7):

    X, y = make_classification(
        n_samples=500,
        n_features=60,
        n_informative=5,
        n_redundant=3,
        n_clusters_per_class=1,
        shuffle=False,
        random_state=seed
    )

    return X, y


# ==========================================
# STEP 2: RFE SUPPORT
# ==========================================

def rfe_support(X, y, n_select):

    estimator = LogisticRegression(max_iter=5000)

    selector = RFE(
        estimator=estimator,
        n_features_to_select=n_select
    )

    selector.fit(X, y)

    return selector.support_


# ==========================================
# STEP 3: RFE RANKING
# ==========================================

def rfe_ranking(X, y):

    estimator = LogisticRegression(max_iter=5000)

    selector = RFE(
        estimator=estimator,
        n_features_to_select=1
    )

    selector.fit(X, y)

    return selector.ranking_


# ==========================================
# STEP 4: RFECV FEATURE SELECTION
# ==========================================

def rfecv_select(X, y):

    estimator = LogisticRegression(max_iter=5000)

    selector = RFECV(
        estimator=estimator,
        cv=5,
        scoring="f1"
    )

    selector.fit(X, y)

    return selector.support_, int(selector.n_features_)


# ==========================================
# STEP 5: CROSS VALIDATION F1
# ==========================================

def f1_cv(X, y):

    estimator = LogisticRegression(max_iter=5000)

    scores = cross_val_score(
        estimator,
        X,
        y,
        cv=5,
        scoring="f1"
    )

    return float(scores.mean())


# ==========================================
# STEP 6: F1 BEFORE AND AFTER SELECTION
# ==========================================

def f1_before_after(X, y, support):

    f1_all = f1_cv(X, y)

    X_selected = X[:, support]

    f1_sel = f1_cv(X_selected, y)

    improvement = f1_sel - f1_all

    return (
        float(f1_all),
        float(f1_sel),
        float(improvement)
    )


# ==========================================
# STEP 7: GENERATE DATA
# ==========================================

X, y = make_data(7)

print("Dataset Shape:")
print("X:", X.shape)
print("y:", y.shape)


# ==========================================
# STEP 8: RFE SUPPORT
# ==========================================

sup = rfe_support(X, y, 8)

print("\nSelected Feature Count:")
print(int(sup.sum()))

print("\nSelected Feature Indices:")
print(np.where(sup)[0])


# ==========================================
# STEP 9: RFE RANKING
# ==========================================

r = rfe_ranking(X, y)

print("\nMinimum Ranking:")
print(int(r.min()))

print("\nNumber of Rank-1 Features:")
print(int((r == 1).sum()))

print("\nComplete Ranking:")
print(r)


# ==========================================
# STEP 10: RFECV FEATURE SELECTION
# ==========================================

mask, n = rfecv_select(X, y)

print("\nRFECV Selected Features:")
print(n)

print("\nRFECV Selected Indices:")
print(np.where(mask)[0])


# ==========================================
# STEP 11: F1 CROSS VALIDATION
# ==========================================

f_all = f1_cv(X, y)

print("\nF1 Score Using All Features:")
print(round(f_all, 3))


# ==========================================
# STEP 12: F1 BEFORE AND AFTER
# ==========================================

f_all, f_sel, imp = f1_before_after(
    X,
    y,
    mask
)

print("\nF1 Before Feature Selection:")
print(round(f_all, 3))

print("\nF1 After Feature Selection:")
print(round(f_sel, 3))

print("\nF1 Improvement:")
print(round(imp, 3))


# ==========================================
# STEP 13: COMPARE SELECTED VS NOISE FEATURES
# ==========================================

X_rfe = X[:, sup]

X_noise = X[:, -8:]

f1_rfe = f1_cv(X_rfe, y)

f1_noise = f1_cv(X_noise, y)

print("\nRFE Selected 8 Features F1:")
print(round(f1_rfe, 3))

print("\nLast 8 Noise Features F1:")
print(round(f1_noise, 3))


# ==========================================
# STEP 14: FINAL VERIFICATION
# ==========================================

print("\nFinal Verification:")

print("RFE selected exactly 8:", sup.sum() == 8)

print("Ranking has one rank-1 feature:", (r == 1).sum() == 1)

print("RFECV selected fewer than 60:", n < 60)

print("RFECV F1 improvement > 0.02:", imp > 0.02)

print("Selected features outperform noise:", f1_rfe > f1_noise)
